# Decorators: Adding Logging, Validation and Timing to Order Functions

In this practice you will write decorators that add logging, input validation and timing to order functions without changing the functions themselves. You will then stack two decorators on one function.

## Background

The retailer's order functions are used across several reports. The team wants every call logged, bad quantities rejected and slow calculations timed. Adding that code to every function would repeat it many times. A decorator adds it once and can be reused.

## Data Description

All the practices in this module use **Retail Orders.csv**: 200 orders from 2025 placed with a small online retailer based in Ireland. The data is fictional and was created for teaching.

| Column Name | Description |
|---|---|
| order_id | Unique order identifier |
| order_date | Date the order was placed (YYYY-MM-DD) |
| delivery_date | Date the order was delivered; blank if not yet delivered |
| customer_name | Customer's name, as typed into the order system |
| city | Delivery city |
| product | Product ordered |
| category | Product category: Electronics, Office or Home |
| quantity | Number of units ordered |
| unit_price | Price of one unit, in euro |
| priority | Order priority: Urgent, Normal or Minor |
| status | Order status: Delivered, Shipped or Cancelled |

## 1. Load the Dataset

We import pandas and read the CSV file into a DataFrame. Keep **Retail Orders.csv** in the same folder as this notebook.

In [1]:
import pandas as pd

df = pd.read_csv("Retail Orders.csv")
df.head()

,order_id,order_date,delivery_date,customer_name,city,product,category,quantity,unit_price,priority,status
0,ORD-1001,2025-01-11,2025-01-19,Tadhg Farrell,Limerick,Filing Cabinet,Office,5,120.0,Minor,Delivered
1,ORD-1002,2025-01-11,2025-01-15,Liam Kelly,Dublin,Whiteboard,Office,3,45.0,Normal,Delivered
2,ORD-1003,2025-01-11,2025-01-13,Sean Ryan,Cork,A4 Notebook,Office,3,4.5,Normal,Delivered
3,ORD-1004,2025-01-12,2025-01-16,Sinead Ryan,Cork,Ergonomic Chair,Office,5,189.0,Normal,Delivered
4,ORD-1005,2025-01-15,2025-01-24,Tomas Lynch,Galway,A4 Notebook,Office,1,4.5,Minor,Delivered


## 2. A Function to Decorate

`order_total()` is the function from Lesson 6.

In [2]:
def order_total(quantity, unit_price):
    return quantity * unit_price

order_total(3, 45.0)

135.0

## 3. Your First Decorator

A decorator is a function that takes a function and returns a new **wrapper** function. The wrapper announces the call, runs the original function, and reports the result.

In [3]:
def announce(func):
    def wrapper(quantity, unit_price):
        print(f"Calling '{func.__name__}' with {quantity} x {unit_price}")
        result = func(quantity, unit_price)
        print(f"Result: {result}")
        return result
    return wrapper

@announce
def order_total(quantity, unit_price):
    return quantity * unit_price

order_total(3, 45.0)

Calling 'order_total' with 3 x 45.0
Result: 135.0


135.0

`@announce` above the function is shorthand for `order_total = announce(order_total)`. We can see that the name now points to the wrapper:

In [4]:
print(order_total.__name__)

wrapper


## 4. A Validation Decorator

Quantities must be between 1 and 10 for a standard order. The validation lives in the decorator, so the calculation stays simple.

In [5]:
def validate_quantity(func):
    def wrapper(quantity, unit_price):
        if quantity <= 0:
            print("Error: quantity must be greater than 0.")
            return
        if quantity > 10:
            print("Error: orders above 10 units need manager approval.")
            return
        return func(quantity, unit_price)
    return wrapper

@validate_quantity
def checked_total(quantity, unit_price):
    return quantity * unit_price

print(checked_total(4, 39.5))
print(checked_total(0, 39.5))
print(checked_total(12, 39.5))

158.0
Error: quantity must be greater than 0.
None
Error: orders above 10 units need manager approval.
None


## 5. A Logging Decorator for Any Function

Using `*args` and `**kwargs` in the wrapper lets one decorator work with functions that take any arguments. `functools.wraps` keeps the original function's name.

In [6]:
import functools

def log_function_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Function '{func.__name__}' called with args: {args}, kwargs: {kwargs}")
        return func(*args, **kwargs)
    return wrapper

## 6. Reuse the Decorator on Two Functions

We calculate the mean and variance of the first five order values. The same decorator logs both functions, including the call that `calculate_variance()` makes to `calculate_mean()`.

In [7]:
@log_function_call
def calculate_mean(values):
    return sum(values) / len(values)

@log_function_call
def calculate_variance(values):
    mean = calculate_mean(values)
    return sum((x - mean) ** 2 for x in values) / len(values)

values = list(df["quantity"][0:5] * df["unit_price"][0:5])
print(values)
print("Variance:", round(calculate_variance(values), 2))

[600.0, 135.0, 13.5, 945.0, 4.5]
Function 'calculate_variance' called with args: ([600.0, 135.0, 13.5, 945.0, 4.5],), kwargs: {}
Function 'calculate_mean' called with args: ([600.0, 135.0, 13.5, 945.0, 4.5],), kwargs: {}
Variance: 138962.34


## 7. A Timing Decorator

`measure_time` records the time before and after the call. We use it on a function that loops through all 200 orders.

In [8]:
import time

def measure_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"'{func.__name__}' executed in {end - start:.4f} seconds.")
        return result
    return wrapper

@measure_time
def total_revenue(data):
    revenue = 0
    for o in data.to_dict("records"):
        if o["status"] != "Cancelled":
            revenue = revenue + o["quantity"] * o["unit_price"]
    return round(revenue, 2)

total_revenue(df)

'total_revenue' executed in 0.0031 seconds.


24279.05

The time shown will differ slightly each time you run the cell.

## 8. A Decorator with Parameters

`check_range(1, 10)` is a function that **builds** a decorator with its own limits, so the same code can be reused with different ranges.

In [9]:
def check_range(min_val, max_val):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(quantity, *args):
            if quantity < min_val or quantity > max_val:
                print(f"Error: quantity {quantity} is outside {min_val}-{max_val}.")
                return None
            return func(quantity, *args)
        return wrapper
    return decorator

## 9. Stack Two Decorators

Decorators are applied from the bottom up: first the range check is wrapped around the function, then the logging decorator is wrapped around that.

In [10]:
@log_function_call
@check_range(1, 10)
def line_total(quantity, unit_price):
    return quantity * unit_price

print(line_total(5, 149.0))
print(line_total(12, 149.0))

Function 'line_total' called with args: (5, 149.0), kwargs: {}
745.0
Function 'line_total' called with args: (12, 149.0), kwargs: {}
Error: quantity 12 is outside 1-10.
None


## Summary

In this practice you:

- wrote a decorator with an inner wrapper function and applied it with `@`
- separated validation from calculation with a validation decorator
- wrote a reusable logging decorator with `*args`, `**kwargs` and `functools.wraps`
- timed a function with a timing decorator
- built a decorator with parameters, and stacked two decorators on one function

In Lesson 9 you will work with the order and delivery dates.